# EEG Feature Engineering — Full Pipeline (v2)

Sanjeev Veeramani (100004303) — Case Study 2 — SRH University Heidelberg

**v2 changes:** RASM formula corrected to power ratio (Zheng & Lu 2015). Per-feature batch sizes tuned for Colab High-RAM.

| Section | Purpose |
|---|---|
| 0 | Drive mount, input verification |
| 1 | SEED per-channel per-subject z-score fix |
| 2 | Butterworth band-pass filter design (5 bands) |
| 3 | Tier 1 extractors: DE, PSD, TEMPORAL |
| 4 | Tier 2 extractors: DASM, RASM, CORR |
| 5 | Tier 3 extractors: PLV, FRACTAL, ENTROPY |
| 6 | Sanity check on 100 segments |
| 7 | Batched runner (per-feature batch size) |
| 8 | DEAP full extraction |
| 9 | SEED full extraction |
| 10 | Per-subject feature z-scoring |
| 11 | Verification |

## Section 0 — Drive mount and verification

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/case_study_2/processed')

print(f'DATA_DIR: {DATA_DIR}')
print(f'Exists:   {DATA_DIR.exists()}\n')

required = [
    'X_deap.npy', 'X_seed.npy',
    'y_deap_valence.npy', 'y_deap_arousal.npy', 'y_deap_3class.npy', 'y_seed.npy',
    'deap_subject_ids.npy', 'seed_subject_ids.npy',
    'common_channels.npy',
]
missing = []
for f in required:
    p = DATA_DIR / f
    if p.exists():
        print(f'  [OK]      {f:35s}  {p.stat().st_size / 1e6:8.1f} MB')
    else:
        print(f'  [MISSING] {f}')
        missing.append(f)
assert not missing, f'Missing: {missing}'

Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/case_study_2/processed
Exists:   True

  [OK]      X_deap.npy                             4991.2 MB
  [OK]      X_seed.npy                             4993.6 MB
  [OK]      y_deap_valence.npy                        1.2 MB
  [OK]      y_deap_arousal.npy                        1.2 MB
  [OK]      y_deap_3class.npy                         1.2 MB
  [OK]      y_seed.npy                                2.4 MB
  [OK]      deap_subject_ids.npy                      1.2 MB
  [OK]      seed_subject_ids.npy                      2.4 MB
  [OK]      common_channels.npy                       0.0 MB


In [2]:
!pip install -q numpy scipy tqdm

In [3]:
import numpy as np
from scipy import signal, stats
from tqdm import tqdm
import time, gc

FS = 128
WINDOW_SAMPLES = 128
N_CHANNELS = 32

BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 14),
    'beta':  (14, 31),
    'gamma': (31, 45),
}
BAND_NAMES = list(BANDS.keys())
N_BANDS = len(BANDS)

channel_names = np.load(DATA_DIR / 'common_channels.npy', allow_pickle=True)
print(f'Channels ({len(channel_names)}): {list(channel_names)}')

Channels (32): [np.str_('FP1'), np.str_('AF3'), np.str_('F3'), np.str_('F7'), np.str_('FC5'), np.str_('FC1'), np.str_('C3'), np.str_('T7'), np.str_('CP5'), np.str_('CP1'), np.str_('P3'), np.str_('P7'), np.str_('PO3'), np.str_('O1'), np.str_('OZ'), np.str_('PZ'), np.str_('FP2'), np.str_('AF4'), np.str_('F4'), np.str_('F8'), np.str_('FC6'), np.str_('FC2'), np.str_('C4'), np.str_('T8'), np.str_('CP6'), np.str_('CP2'), np.str_('P4'), np.str_('P8'), np.str_('PO4'), np.str_('O2'), np.str_('FZ'), np.str_('CZ')]


## Section 1 — SEED per-channel per-subject z-score fix

SEED raw currently has per-channel std spanning [0.4, 15.8] due to global normalization. Fix: per-subject, per-channel z-score across all time samples. Skip this section if X_seed.npy has already been fixed.

In [4]:
# Skip if already fixed (backup file exists)
if (DATA_DIR / 'X_seed_original.npy').exists():
    print('SEED already fixed (X_seed_original.npy backup exists). Skipping Section 1.')
else:
    print('BEFORE FIX:')
    X_seed_mmap = np.load(DATA_DIR / 'X_seed.npy', mmap_mode='r')
    sample = np.asarray(X_seed_mmap[:1000])
    print(f'  shape={X_seed_mmap.shape}')
    print(f'  overall std={sample.std():.4f}')
    print(f'  per-channel std range=[{sample.std(axis=(0,2)).min():.3f}, {sample.std(axis=(0,2)).max():.3f}]')
    del sample

    import shutil
    print('\nBacking up X_seed.npy -> X_seed_original.npy')
    shutil.copy(DATA_DIR / 'X_seed.npy', DATA_DIR / 'X_seed_original.npy')

    X_seed = np.load(DATA_DIR / 'X_seed.npy')
    seed_sids = np.load(DATA_DIR / 'seed_subject_ids.npy')
    print(f'\nNormalizing per subject per channel...')
    X_seed_fixed = np.empty_like(X_seed, dtype=np.float32)
    for sid in tqdm(np.unique(seed_sids), desc='Subjects'):
        mask = seed_sids == sid
        Xs = X_seed[mask]
        Xs_flat = Xs.transpose(0, 2, 1).reshape(-1, 32)
        mu = Xs_flat.mean(axis=0)
        sigma = Xs_flat.std(axis=0) + 1e-8
        X_seed_fixed[mask] = ((Xs - mu[None, :, None]) / sigma[None, :, None]).astype(np.float32)

    np.save(DATA_DIR / 'X_seed.npy', X_seed_fixed)
    print('\nAFTER FIX:')
    sample = X_seed_fixed[:1000]
    print(f'  overall std={sample.std():.4f}')
    print(f'  per-channel std range=[{sample.std(axis=(0,2)).min():.3f}, {sample.std(axis=(0,2)).max():.3f}]')
    del X_seed, X_seed_fixed
    gc.collect()

SEED already fixed (X_seed_original.npy backup exists). Skipping Section 1.


## Section 2 — Butterworth band-pass filter design

In [5]:
def make_band_filter(low, high, fs, order=4):
    nyq = fs / 2
    low_n = max(low / nyq, 1e-4)
    high_n = min(high / nyq, 0.9999)
    return signal.butter(order, [low_n, high_n], btype='band')

BAND_FILTERS = {name: make_band_filter(lo, hi, FS) for name, (lo, hi) in BANDS.items()}
print('Band filters:', list(BAND_FILTERS.keys()))

Band filters: ['delta', 'theta', 'alpha', 'beta', 'gamma']


## Section 3 — Tier 1 extractors

**DE:** 0.5·log(2πe·σ²) per channel per band. Output (N, 160).  
**PSD:** log(mean band power) via Welch. Output (N, 160).  
**TEMPORAL:** mean, variance, skewness, kurtosis per channel. Output (N, 128).

In [6]:
def compute_DE(segments):
    N, C, T = segments.shape
    out = np.zeros((N, C, N_BANDS), dtype=np.float32)
    const = 0.5 * np.log(2 * np.pi * np.e)
    for b, name in enumerate(BAND_NAMES):
        b_coef, a_coef = BAND_FILTERS[name]
        filtered = signal.filtfilt(b_coef, a_coef, segments, axis=-1)
        var = filtered.var(axis=-1) + 1e-10
        out[:, :, b] = const + 0.5 * np.log(var).astype(np.float32)
    return out.reshape(N, -1)

def compute_PSD(segments):
    N, C, T = segments.shape
    freqs, psd = signal.welch(segments, fs=FS, nperseg=T, axis=-1)
    out = np.zeros((N, C, N_BANDS), dtype=np.float32)
    for b, name in enumerate(BAND_NAMES):
        lo, hi = BANDS[name]
        idx = (freqs >= lo) & (freqs <= hi)
        band_power = psd[:, :, idx].mean(axis=-1)
        out[:, :, b] = np.log(band_power + 1e-10).astype(np.float32)
    return out.reshape(N, -1)

def compute_TEMPORAL(segments):
    means = segments.mean(axis=-1)
    vars_ = segments.var(axis=-1)
    skews = stats.skew(segments, axis=-1)
    kurts = stats.kurtosis(segments, axis=-1)
    out = np.stack([means, vars_, skews, kurts], axis=-1).astype(np.float32)
    return out.reshape(out.shape[0], -1)

print('Tier 1: DE, PSD, TEMPORAL')

Tier 1: DE, PSD, TEMPORAL


## Section 4 — Tier 2 extractors

**DASM:** log-power(left) − log-power(right) per hemispheric pair per band.  
**RASM:** power(left) / power(right) per hemispheric pair per band (Zheng & Lu 2015).  
**CORR:** Pearson correlation upper triangle of channel × channel matrix. Output (N, 496).

In [7]:
import re

def find_hemispheric_pairs(channel_names):
    channel_names = list(channel_names)
    pairs = []
    used = set()
    for name in channel_names:
        if name in used:
            continue
        m = re.match(r'([A-Za-z]+)(\d+)([A-Za-z]*)', name)
        if not m:
            continue
        prefix, num_str, suffix = m.group(1), m.group(2), m.group(3)
        num = int(num_str)
        counterpart = f'{prefix}{num + 1 if num % 2 == 1 else num - 1}{suffix}'
        if counterpart in channel_names and counterpart != name:
            left = name if num % 2 == 1 else counterpart
            right = counterpart if num % 2 == 1 else name
            pairs.append((left, right, channel_names.index(left), channel_names.index(right)))
            used.add(left); used.add(right)
    return pairs

HEMISPHERIC_PAIRS = find_hemispheric_pairs(channel_names)
PAIR_LEFT_IDX = np.array([p[2] for p in HEMISPHERIC_PAIRS])
PAIR_RIGHT_IDX = np.array([p[3] for p in HEMISPHERIC_PAIRS])
N_PAIRS = len(HEMISPHERIC_PAIRS)
print(f'Hemispheric pairs: {N_PAIRS}')
for l, r, il, ir in HEMISPHERIC_PAIRS:
    print(f'  {l} ({il}) <-> {r} ({ir})')

Hemispheric pairs: 14
  FP1 (0) <-> FP2 (16)
  AF3 (1) <-> AF4 (17)
  F3 (2) <-> F4 (18)
  F7 (3) <-> F8 (19)
  FC5 (4) <-> FC6 (20)
  FC1 (5) <-> FC2 (21)
  C3 (6) <-> C4 (22)
  T7 (7) <-> T8 (23)
  CP5 (8) <-> CP6 (24)
  CP1 (9) <-> CP2 (25)
  P3 (10) <-> P4 (26)
  P7 (11) <-> P8 (27)
  PO3 (12) <-> PO4 (28)
  O1 (13) <-> O2 (29)


In [8]:
def _band_log_power(segments):
    N, C, T = segments.shape
    out = np.zeros((N, C, N_BANDS), dtype=np.float32)
    for b, name in enumerate(BAND_NAMES):
        b_coef, a_coef = BAND_FILTERS[name]
        filtered = signal.filtfilt(b_coef, a_coef, segments, axis=-1)
        out[:, :, b] = np.log(filtered.var(axis=-1) + 1e-10).astype(np.float32)
    return out

def compute_DASM(segments):
    lp = _band_log_power(segments)
    dasm = lp[:, PAIR_LEFT_IDX, :] - lp[:, PAIR_RIGHT_IDX, :]
    return dasm.reshape(dasm.shape[0], -1)

def compute_RASM(segments):
    lp = _band_log_power(segments)
    left = np.exp(lp[:, PAIR_LEFT_IDX, :])
    right = np.exp(lp[:, PAIR_RIGHT_IDX, :]) + 1e-10
    rasm = (left / right).astype(np.float32)
    return rasm.reshape(rasm.shape[0], -1)

def compute_CORR(segments):
    N, C, T = segments.shape
    triu_i, triu_j = np.triu_indices(C, k=1)
    out = np.zeros((N, len(triu_i)), dtype=np.float32)
    for n in range(N):
        cm = np.corrcoef(segments[n])
        out[n] = cm[triu_i, triu_j]
    return out

print(f'Tier 2: DASM ({N_PAIRS * N_BANDS}), RASM ({N_PAIRS * N_BANDS}), CORR ({N_CHANNELS * (N_CHANNELS - 1) // 2})')

Tier 2: DASM (70), RASM (70), CORR (496)


## Section 5 — Tier 3 extractors

**PLV:** |mean_t(exp(i(φ_L − φ_R)))| per band per channel pair. Output (N, 2480).  
**FRACTAL:** Higuchi Fractal Dimension per channel, k_max=10. Output (N, 32).  
**ENTROPY:** Permutation Entropy (Bandt & Pompe 2002), m=3, delay=1. Output (N, 32).

In [9]:
def compute_PLV(segments):
    N, C, T = segments.shape
    triu_i, triu_j = np.triu_indices(C, k=1)
    n_pairs = len(triu_i)
    out = np.zeros((N, n_pairs, N_BANDS), dtype=np.float32)
    for b_idx, band_name in enumerate(BAND_NAMES):
        b, a = BAND_FILTERS[band_name]
        filtered = signal.filtfilt(b, a, segments, axis=-1)
        analytic = signal.hilbert(filtered, axis=-1)
        phases = np.angle(analytic)
        phase_diff = phases[:, triu_i, :] - phases[:, triu_j, :]
        plv = np.abs(np.mean(np.exp(1j * phase_diff), axis=-1))
        out[:, :, b_idx] = plv.astype(np.float32)
    return out.reshape(N, -1)

def _higuchi_fd(x, k_max=10):
    N = len(x)
    L = np.zeros(k_max)
    for k in range(1, k_max + 1):
        Lk = 0
        for m in range(k):
            n_steps = (N - m - 1) // k
            if n_steps < 1:
                continue
            sub = x[m::k][:n_steps + 1]
            diffs = np.abs(np.diff(sub))
            Lmk = (diffs.sum() * (N - 1) / (n_steps * k)) / k
            Lk += Lmk / k
        L[k - 1] = Lk
    log_k = np.log(np.arange(1, k_max + 1))
    log_L = np.log(L + 1e-10)
    return -np.polyfit(log_k, log_L, 1)[0]

def compute_FRACTAL(segments, k_max=10):
    N, C, T = segments.shape
    out = np.zeros((N, C), dtype=np.float32)
    for n in range(N):
        for c in range(C):
            out[n, c] = _higuchi_fd(segments[n, c], k_max=k_max)
    return out

def _perm_entropy(x, m=3, delay=1):
    N = len(x)
    n_vecs = N - (m - 1) * delay
    if n_vecs <= 0:
        return 0.0
    idx = np.arange(m) * delay
    embeddings = x[np.arange(n_vecs)[:, None] + idx[None, :]]
    ranks = np.argsort(np.argsort(embeddings, axis=1), axis=1)
    codes = np.zeros(n_vecs, dtype=int)
    for i in range(m):
        codes = codes * m + ranks[:, i]
    _, counts = np.unique(codes, return_counts=True)
    p = counts / n_vecs
    return float(-np.sum(p * np.log(p + 1e-12)))

def compute_ENTROPY(segments, m=3, delay=1):
    N, C, T = segments.shape
    out = np.zeros((N, C), dtype=np.float32)
    for n in range(N):
        for c in range(C):
            out[n, c] = _perm_entropy(segments[n, c], m=m, delay=delay)
    return out

print('Tier 3: PLV, FRACTAL, ENTROPY')

Tier 3: PLV, FRACTAL, ENTROPY


## Section 6 — Sanity check

Verify all 9 extractors on 100 DEAP segments. Expected ranges: PLV ∈ [0,1], FRACTAL ∈ [1,2], ENTROPY > 0, RASM > 0 (ratio, not log-ratio, so different from DASM).

In [10]:
X_deap = np.load(DATA_DIR / 'X_deap.npy', mmap_mode='r')
sample = np.asarray(X_deap[:100])
print(f'Sample: {sample.shape}\n')

extractors = [
    ('DE', compute_DE), ('PSD', compute_PSD), ('TEMPORAL', compute_TEMPORAL),
    ('DASM', compute_DASM), ('RASM', compute_RASM), ('CORR', compute_CORR),
    ('PLV', compute_PLV), ('FRACTAL', compute_FRACTAL), ('ENTROPY', compute_ENTROPY),
]

for name, fn in extractors:
    t0 = time.time()
    feat = fn(sample)
    dt = time.time() - t0
    print(f'  {name:9s} shape={str(feat.shape):15s} range=[{feat.min():8.3f}, {feat.max():8.3f}]  mean={feat.mean():7.3f}  time={dt:6.2f}s  NaN={np.isnan(feat).any()}')

Sample: (100, 32, 128)

  DE        shape=(100, 160)      range=[  -2.478,    1.849]  mean=  0.268  time=  0.13s  NaN=False
  PSD       shape=(100, 160)      range=[ -10.509,   -0.586]  mean= -4.510  time=  0.02s  NaN=False
  TEMPORAL  shape=(100, 128)      range=[  -2.573,   13.055]  mean=  0.232  time=  0.03s  NaN=False
  DASM      shape=(100, 70)       range=[  -4.435,    4.576]  mean=  0.047  time=  0.11s  NaN=False
  RASM      shape=(100, 70)       range=[   0.012,   97.141]  mean=  1.674  time=  0.12s  NaN=False
  CORR      shape=(100, 496)      range=[  -0.932,    0.968]  mean= -0.029  time=  0.02s  NaN=False
  PLV       shape=(100, 2480)     range=[   0.001,    1.000]  mean=  0.476  time=  2.01s  NaN=False
  FRACTAL   shape=(100, 32)       range=[   1.363,    1.927]  mean=  1.708  time=  1.65s  NaN=False
  ENTROPY   shape=(100, 32)       range=[   1.343,    1.735]  mean=  1.582  time=  0.29s  NaN=False


## Section 7 — Batched extraction runner (per-feature batch size)

Batch sizes tuned for Colab High-RAM (~53 GB). Larger batches reduce Python overhead. PLV uses smaller batches due to complex Hilbert-transform intermediates. Progressive save writes each feature to Drive immediately. skip_existing=True resumes from interrupted runs.

In [11]:
BATCH_SIZES = {
    'DE':       20000,
    'PSD':      20000,
    'TEMPORAL': 20000,
    'DASM':     20000,
    'RASM':     20000,
    'CORR':     5000,
    'PLV':      3000,
    'FRACTAL':  5000,
    'ENTROPY':  10000,
}

def extract_in_batches(X_source, feature_fn, feature_name, batch):
    N = X_source.shape[0]
    outs = []
    for i in tqdm(range(0, N, batch), desc=f'{feature_name} (batch={batch})'):
        chunk = np.asarray(X_source[i:i + batch])
        outs.append(feature_fn(chunk))
    return np.concatenate(outs, axis=0)

def run_dataset(dataset_name, features, skip_existing=True):
    print(f'\n=== {dataset_name.upper()} ===')
    X_source = np.load(DATA_DIR / f'X_{dataset_name}.npy', mmap_mode='r')
    print(f'Source: {X_source.shape}\n')
    for name, fn in features:
        out_path = DATA_DIR / f'X_{dataset_name}_{name}.npy'
        if skip_existing and out_path.exists():
            print(f'[SKIP] {out_path.name}')
            continue
        batch = BATCH_SIZES.get(name, 2000)
        t0 = time.time()
        feats = extract_in_batches(X_source, fn, name, batch=batch)
        dt = (time.time() - t0) / 60
        np.save(out_path, feats)
        print(f'[SAVED] {out_path.name}  shape={feats.shape}  time={dt:.1f} min')
        del feats
        gc.collect()

print('Runner ready with per-feature batching.')

Runner ready with per-feature batching.


## Section 8 — DEAP full extraction

**Already saved before interrupt:** DE, PSD, TEMPORAL, DASM, RASM, CORR — will be skipped.  
**Remaining:** PLV, FRACTAL, ENTROPY (~1.5 hours combined).

In [14]:
ALL_FEATURES = [
    ('DE', compute_DE), ('PSD', compute_PSD), ('TEMPORAL', compute_TEMPORAL),
    ('DASM', compute_DASM), ('RASM', compute_RASM), ('CORR', compute_CORR),
    ('PLV', compute_PLV), ('FRACTAL', compute_FRACTAL), ('ENTROPY', compute_ENTROPY),
]

run_dataset('deap', ALL_FEATURES, skip_existing=True)


=== DEAP ===
Source: (152320, 32, 128)

[SKIP] X_deap_DE.npy
[SKIP] X_deap_PSD.npy
[SKIP] X_deap_TEMPORAL.npy
[SKIP] X_deap_DASM.npy
[SKIP] X_deap_RASM.npy
[SKIP] X_deap_CORR.npy


PLV (batch=3000): 100%|██████████| 51/51 [46:11<00:00, 54.35s/it]


[SAVED] X_deap_PLV.npy  shape=(152320, 2480)  time=46.2 min


FRACTAL (batch=5000): 100%|██████████| 31/31 [38:14<00:00, 74.01s/it]


[SAVED] X_deap_FRACTAL.npy  shape=(152320, 32)  time=38.2 min


ENTROPY (batch=10000): 100%|██████████| 16/16 [05:58<00:00, 22.41s/it]


[SAVED] X_deap_ENTROPY.npy  shape=(152320, 32)  time=6.0 min


## Section 9 — SEED full extraction (uses fixed raw)

All 9 features from scratch on 304785 SEED segments (~3.5 hours).

In [15]:
run_dataset('seed', ALL_FEATURES, skip_existing=True)


=== SEED ===
Source: (304785, 32, 128)



DE (batch=20000): 100%|██████████| 16/16 [06:11<00:00, 23.19s/it]


[SAVED] X_seed_DE.npy  shape=(304785, 160)  time=6.2 min


PSD (batch=20000): 100%|██████████| 16/16 [00:34<00:00,  2.17s/it]


[SAVED] X_seed_PSD.npy  shape=(304785, 160)  time=0.6 min


...

[SAVED] X_seed_PLV.npy  shape=(304785, 2480)  time=91.9 min


FRACTAL (batch=5000): 100%|██████████| 61/61 [1:16:45<00:00, 75.50s/it]


[SAVED] X_seed_FRACTAL.npy  shape=(304785, 32)  time=76.8 min


ENTROPY (batch=10000): 100%|██████████| 31/31 [11:56<00:00, 23.12s/it]


[SAVED] X_seed_ENTROPY.npy  shape=(304785, 32)  time=11.9 min


## Section 10 — Per-subject feature z-scoring

Each feature file re-scaled per subject. Improves classical baseline LOSO accuracy by 2–5 pp. Output: *_zscored.npy for each of 18 files.

In [16]:
def zscore_per_subject(X, subject_ids):
    X_out = np.empty_like(X, dtype=np.float32)
    for sid in np.unique(subject_ids):
        mask = subject_ids == sid
        Xs = X[mask]
        mu = Xs.mean(axis=0)
        sigma = Xs.std(axis=0) + 1e-8
        X_out[mask] = ((Xs - mu) / sigma).astype(np.float32)
    return X_out

FEATURE_NAMES = ['DE', 'PSD', 'TEMPORAL', 'DASM', 'RASM', 'CORR', 'PLV', 'FRACTAL', 'ENTROPY']

for dataset in ['deap', 'seed']:
    sids = np.load(DATA_DIR / f'{dataset}_subject_ids.npy')
    print(f'\n[{dataset.upper()}]')
    for name in FEATURE_NAMES:
        raw_path = DATA_DIR / f'X_{dataset}_{name}.npy'
        z_path = DATA_DIR / f'X_{dataset}_{name}_zscored.npy'
        if z_path.exists():
            print(f'  [SKIP] {z_path.name}')
            continue
        if not raw_path.exists():
            print(f'  [MISSING] {raw_path.name}')
            continue
        X = np.load(raw_path)
        Xz = zscore_per_subject(X, sids)
        np.save(z_path, Xz)
        print(f'  [SAVED] {z_path.name}')
        del X, Xz
        gc.collect()


[DEAP]
  [SKIP] X_deap_DE_zscored.npy
  [SKIP] X_deap_PSD_zscored.npy
  [SKIP] X_deap_TEMPORAL_zscored.npy
  [SAVED] X_deap_DASM_zscored.npy
  [SAVED] X_deap_RASM_zscored.npy
  [SAVED] X_deap_CORR_zscored.npy
  [SAVED] X_deap_PLV_zscored.npy
  [SAVED] X_deap_FRACTAL_zscored.npy
  [SAVED] X_deap_ENTROPY_zscored.npy

[SEED]
  [SAVED] X_seed_DE_zscored.npy
  [SAVED] X_seed_PSD_zscored.npy
  [SAVED] X_seed_TEMPORAL_zscored.npy
  [SAVED] X_seed_DASM_zscored.npy
  [SAVED] X_seed_RASM_zscored.npy
  [SAVED] X_seed_CORR_zscored.npy
  [SAVED] X_seed_PLV_zscored.npy
  [SAVED] X_seed_FRACTAL_zscored.npy
  [SAVED] X_seed_ENTROPY_zscored.npy


## Section 11 — Verification

Expected: 36 files ([OK]) — 9 features × 2 datasets × 2 variants (raw + z-scored).

In [17]:
for dataset in ['deap', 'seed']:
    print(f'\n[{dataset.upper()}]')
    for tier, names in [(1, ['DE', 'PSD', 'TEMPORAL']),
                        (2, ['DASM', 'RASM', 'CORR']),
                        (3, ['PLV', 'FRACTAL', 'ENTROPY'])]:
        for name in names:
            for suffix in ['', '_zscored']:
                p = DATA_DIR / f'X_{dataset}_{name}{suffix}.npy'
                if p.exists():
                    f = np.load(p, mmap_mode='r')
                    print(f'  [OK] T{tier} {name}{suffix:9s} shape={str(f.shape):20s} size={p.stat().st_size / 1e6:6.1f} MB')
                else:
                    print(f'  [--] T{tier} {name}{suffix:9s} missing')


[DEAP]
  [OK] T1 DE          shape=(152320, 160)        size=  97.5 MB
  [OK] T1 DE_zscored  shape=(152320, 160)        size=  97.5 MB
  [OK] T1 PSD          shape=(152320, 160)        size=  97.5 MB
  [OK] T1 PSD_zscored  shape=(152320, 160)        size=  97.5 MB
  [OK] T1 TEMPORAL          shape=(152320, 128)        size=  78.0 MB
  [OK] T1 TEMPORAL_zscored  shape=(152320, 128)        size=  78.0 MB
  [OK] T2 DASM          shape=(152320, 70)         size=  42.6 MB
  [OK] T2 DASM_zscored  shape=(152320, 70)         size=  42.6 MB
  [OK] T2 RASM          shape=(152320, 70)         size=  42.6 MB
  [OK] T2 RASM_zscored  shape=(152320, 70)         size=  42.6 MB
  [OK] T2 CORR          shape=(152320, 496)        size= 302.2 MB
  [OK] T2 CORR_zscored  shape=(152320, 496)        size= 302.2 MB
  [OK] T3 PLV          shape=(152320, 2480)       size=1511.0 MB
  [OK] T3 PLV_zscored  shape=(152320, 2480)       size=1511.0 MB
  [OK] T3 FRACTAL          shape=(152320, 32)         size=  19.5 MB